In [0]:
bronze_orders = spark.table("workspace.olist_bronze.orders")
bronze_payments = spark.table("workspace.olist_bronze.payments")
bronze_products = spark.table("workspace.olist_bronze.products")
bronze_items = spark.table("workspace.olist_bronze.order_items")
bronze_reviews = spark.table("workspace.olist_bronze.reviews")
bronze_category_map = spark.table("workspace.olist_bronze.category_map")
bronze_seller = spark.table("workspace.olist_bronze.sellers")
bronze_geolocation = spark.table("workspace.olist_bronze.geolocation")
bronze_customers = spark.table("workspace.olist_bronze.customers")


In [0]:

from pyspark.sql.types import DoubleType, TimestampType
from pyspark.sql.window import Window 

from pyspark.sql.functions import *

Order table 


In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS olist_silver")
spark.sql("USE olist_silver")
spark.sql("SELECT current_database()").show()

In [0]:
bronze_orders.printSchema()
# Schema seems good all the column are defined perfectly 
 
 #Removing duplicates 
w_oc = Window.partitionBy("order_id").orderBy(desc('order_purchase_timestamp'))
b_o_df = bronze_orders.withColumn("rank" , row_number().over(w_oc))\
    .filter(col("rank")==1)\
    .drop("rank")

# adding delivery columns
b_o_df = b_o_df.withColumn("Late_delivery" , \
    when(col('order_delivered_customer_date') > col('order_estimated_delivery_date'), \
        'Yes').otherwise('No'))

# validation befor loading the data into Silver table 
print(f"Bronze orders: {spark.table('workspace.olist_bronze.orders').count():,}")
print(f"Silver orders: {b_o_df.count():,}")
assert b_o_df.filter(col('order_id').isNull()).count() == 0,"Null Id's found"

#pusing orders data to Silver table 
b_o_df.writeTo("workspace.olist_silver.orders")\
    .using("delta")\
    .createOrReplace()


print("Orders Data loaded into silver table")

Customer Table 

In [0]:
bronze_customers.printSchema()
#schmea seems good


#remove the spaces in the City and State column 

brz_cust = bronze_customers.withColumn('customer_city' , trim(col('customer_city')))\
                            .withColumn('customer_state', trim(col('customer_state')))


#validate the data

print(f"Bronze Customers: {bronze_customers.count():,}")
print(f"Silver Customers: {brz_cust.count():,}")

assert brz_cust.filter(col('customer_id').isNull()).count() == 0,"Null Id's found"

#load the data into Silver table
brz_cust.writeTo("workspace.olist_silver.customers")\
    .using("delta")\
    .createOrReplace()
print("Customers Data loaded into silver table")

Product Table Igestion into Silver


In [0]:
bronze_products.printSchema()

 # fill the null values with 'Unknown'
brz_null_fill = bronze_products.fillna({'product_category_name': 'unknown'})

#Join with category map to get the english name of the products

joined_tbls = brz_null_fill.join(bronze_category_map , 'product_category_name' ,how= 'left')

bz_prds = joined_tbls.select(brz_null_fill["*"], coalesce(trim(col("product_category_name_english")), \
    lit("unknown")).alias("category_english")
).drop("product_category_name_english")

# cast the columns to double

col_to_cast = ['product_weight_g', 'product_length_cm', 
               'product_height_cm', 'product_width_cm']
for c in col_to_cast:
    bz_prds = bz_prds.withColumn(c, bz_prds[c].cast('double'))

#validate the data
print(f"Bronze Products: {spark.table('workspace.olist_bronze.products').count():,}")
print(f"Silver Products: {bz_prds.count():,}")
assert bz_prds.filter(col('product_id').isNull()).count() == 0,"Null Id's found"


#pushing orders data to Silver table
bz_prds.writeTo("workspace.olist_silver.products")\
    .using("delta")\
    .createOrReplace()
print("Produts Data loaded into silver table")




Items Table


In [0]:
bronze_items.printSchema()

# schema seems good

# add new column
bz_oi = bronze_items.withColumn('total_cost', round(col('price')+col('freight_value'),2))


# null check 

def check(cl):
    return bz_oi.filter(col(cl).isNull()).count()
cols = ['order_id','product_id','seller_id']
for c in cols:
    a = check(c)
    if a > 0:
        assert a == 0, "Null Id's found"
    else:
        print(f"No Null Id's found in {c}")

#Validate the data 
print(f"Bronze Items: {spark.table('workspace.olist_bronze.order_items').count():,}")
print(f"Silver Items: {bz_oi.count():,}")

#pushing order_items  data to Silver table

bz_oi.writeTo("workspace.olist_silver.order_items")\
    .using("delta")\
    .createOrReplace()
print("order_items Data loaded into silver table")


Payments Table


In [0]:
bronze_payments.printSchema()
# schema is ok no need for changes



# Null Check 
def null_check(cl):
    return bronze_payments.filter(col(cl).isNull()).count()
    pass

col_m = ['order_id','payment_type','payment_value']

for c in col_m:
    a= null_check(c)
    if a > 0:
        assert a ==0 ,"Null Id's Found in {c}"
    else:
        print(f"No Null Id's found in {c}")

# add column is_installment 

bz_pt = bronze_payments.withColumn('is_installment',
                                   when(col('payment_installments') > 1, True).otherwise(False))


# validation 

print(f"Bronze Payment: {spark.table('workspace.olist_bronze.payments').count():,}")
print(f"Silver Payments: {bz_pt.count():,}")

# Push the data into Silver table 

bz_pt.writeTo("workspace.olist_silver.payments").using("delta")\
    .createOrReplace()
print("Payments Data loaded into silver table")



Reviews Table 

In [0]:
bronze_reviews.printSchema()
# Schema seems ok no need to do anything 

# add new column 

bz_rev = bronze_reviews.withColumn('has_comment', \
    when(col("review_comment_message").isNull(), False).otherwise(True))


# cehck fro Null vlaues
def check_nl(cl):
    return bz_rev.filter(col(cl).isNull()).count()
    pass

col_s = ['order_id','review_score']

for c in col_s:
    a = check_nl(c)
    if a > 0:
        assert a == 0, "Null Id's found in column {c}"
    else:
        print(f"No Null Id's found in {c}")


# change the null values

bz_rev_fin= bz_rev.withColumn('review_comment_title' , coalesce(trim(col(c)),\
    lit('No Comment Title')))

bz_rev_fin= bz_rev_fin.withColumn('review_comment_message' , coalesce(trim(col(c)),\
    lit('No Comment')))


# validation 
print(f"Bronze Reviews: {spark.table('workspace.olist_bronze.reviews').count():,}")
print(f"Silver Reviews: {bz_rev_fin.count():,}")

# Push the Reviews data into Silver table 

bz_rev_fin.writeTo("workspace.olist_silver.reviews").using("delta")\
    .createOrReplace()
print("Reviews Data loaded into silver table")

    


Seller Table 

In [0]:
bronze_seller.printSchema()

#schmea seems good


#remove the spaces in the City and State column 

brz_seller = bronze_seller.withColumn('seller_city' , trim(col('seller_city')))\
                            .withColumn('seller_state', trim(col('seller_state')))


#validate the data

print(f"Bronze seller: {bronze_seller.count():,}")
print(f"Silver seller: {brz_seller.count():,}")

assert brz_seller.filter(col('seller_id').isNull()).count() == 0,"Null Id's found"

#load the data into Silver table
brz_seller.writeTo("workspace.olist_silver.seller")\
    .using("delta")\
    .createOrReplace()
print("seller Data loaded into silver table")

In [0]:
bronze_geolocation.printSchema()
# schema looks good


# deduplication 

brz_geo = bronze_geolocation.groupBy('geolocation_zip_code_prefix')\
                            .agg(\
                                avg('geolocation_lat').alias('lat'),
                                avg('geolocation_lng').alias('long'),
                                first("geolocation_city").alias('city'),
                                first("geolocation_state").alias('state')
                            )

#validate the data

print(f"Bronze geolocation: {bronze_geolocation.count():,}")
print(f"Silver geolocation: {brz_geo.count():,}")

assert brz_geo.filter(col('geolocation_zip_code_prefix').isNull()).count() == 0,"Null Id's found"

#load the data into Silver table
brz_geo.writeTo("workspace.olist_silver.geolocation")\
    .using("delta")\
    .createOrReplace()
print("geolocation Data loaded into silver table")


In [0]:
bronze_category_map.printSchema()
# schema looks good


# deduplication 

brz_cat = bronze_category_map.withColumn("product_category_name", trim(col("product_category_name"))) \
    .withColumn("product_category_name_english", trim(col("product_category_name_english")))

#validate the data

print(f"Bronze category_map: {bronze_category_map.count():,}")
print(f"Silver category_map: {brz_cat.count():,}")

assert brz_cat.filter(col('product_category_name').isNull()).count() == 0,"Null Id's found"

#load the data into Silver table
brz_cat.writeTo("workspace.olist_silver.category_map").using("delta")\
    .createOrReplace()
print("category_map Data loaded into silver table")



The Main Join 

In [0]:
items = spark.table("workspace.olist_silver.order_items")
orders = spark.table("workspace.olist_silver.orders")
products = spark.table("workspace.olist_silver.products")
customers = spark.table("workspace.olist_silver.customers")
sellers = spark.table("workspace.olist_silver.seller")
geo = spark.table("workspace.olist_silver.geolocation")
payments = spark.table("workspace.olist_silver.payments")
reviews = spark.table("workspace.olist_silver.reviews")
category_map = spark.table("workspace.olist_silver.category_map")



In [0]:
def drop_audit(df):
    return df.drop("_ingest_at", "_ingested_at", "_source")
    # drop both versions — covers any naming inconsistency


geo_clean = drop_audit(geo)

payments_agg = drop_audit(payments) \
    .groupBy("order_id") \
    .agg(sum("payment_value").alias("total_order_payment_value"))

reviews_agg = drop_audit(reviews) \
    .groupBy("order_id") \
    .agg(avg("review_score").alias("avg_review_score"))

master_df = drop_audit(items) \
    .join(drop_audit(orders),    "order_id",    "left") \
    .join(drop_audit(customers), "customer_id", "left") \
    .join(drop_audit(products),  "product_id",  "left") \
    .join(drop_audit(sellers),   "seller_id",   "left") \
    .join(payments_agg,          "order_id",    "left") \
    .join(reviews_agg,           "order_id",    "left")

final_silver_table = master_df.join(
    geo_clean,
    master_df["customer_zip_code_prefix"] == geo_clean["geolocation_zip_code_prefix"],
    "left"
).drop("geolocation_zip_code_prefix") \
 .withColumn("_ingested_at", current_timestamp()) \
 .withColumn("_source", lit("master_sales_data"))

final_silver_table.writeTo("workspace.olist_silver.master_sales_data") \
    .using("delta") \
    .createOrReplace()

print(f"Columns: {len(final_silver_table.columns)}")
print(f"Row Count: {final_silver_table.count():,}")